### Calculation of EarthQuake's Impact Damage Potential

### Load The Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('C:\\Users\\win10\\Desktop\\Project_Oct25\\impactsenseAI-Infosys-Intern-project\\data\\preprocessed_earthquake_data.csv')
df.info()

### 1. Magnitude Type Prioritization and Standardization

In [ ]:
def convert_to_mw(mag_value, mag_type):
    """Convert various earthquake magnitude types to moment magnitude (Mw)."""
    # Reference conversions based on empirical relationships
    if mag_type.lower().startswith('mw'):
        # Moment Magnitude is already in the desired format
        return mag_value
    elif mag_type.lower() == 'ms':
        # Surface Wave Magnitude to Moment Magnitude
        return 1.05 * mag_value - 0.2
    elif mag_type.lower() == 'mb':
        # Body Wave Magnitude to Moment Magnitude
        if mag_value > 6.5:
            return 6.5 + (mag_value - 6.5) * 1.5 # Correction for saturation
        # For mb <= 6.5
        return 0.67 * mag_value + 3.2
    # Add more conversions as needed
    elif mag_type.lower() == 'ml':
        return 1.2 * mag_value - 1.0
    else:
        # Unknown magnitude type
        return np.nan

In [ ]:
# Apply the conversion to the DataFrame
import numpy as np
df['Mw'] = df.apply(lambda row: convert_to_mw(row['mag'], row['magType']), axis=1)
df[['mag', 'magType', 'Mw']].head(10)

#### 2. Calculate Damage Potential Using HAZUS-Style Formula

In [ ]:
def calculate_damage_potential_hazus(magnitude, depth):
    """Calculate earthquake damage potential using HAZUS methodology."""
    actual_depth = max(abs(depth), 1.0)  # Ensure depth is at least 1 km to avoid log(0)
    log_pga = magnitude - 3.5 * np.log10(actual_depth + 7) + 1.8 # HAZUS empirical formula
    pga = 10 ** log_pga  # Convert log10(PGA) to PGA in g
    # Calculate damage potential score (0 to 10 scale)
    damage_potential = min(10.0, max(0.0, 2.5 * np.log10(pga + 0.01) + 7.5))
    # Return the final damage potential score
    return damage_potential

In [ ]:
# Apply the conversion to the DataFrame
df['damage_potential'] = df.apply(lambda row: calculate_damage_potential_hazus(row['Mw'], row['depth']), axis=1)
df[['Mw', 'depth', 'damage_potential']].head(10)